# 31  News coverage of the shortlist

**The question.** Vishal's earlier work asked whether national journalism reaches UK SMEs
and found that it does not: 0 of 96 companies survived disambiguation, while a Tesco
control returned 1,209 articles, which proved the plumbing worked and the absence was real.

That test used a **stress grid**, 96 companies picked to be awkward. This notebook asks the
same question of the population that actually matters: **the 398 companies Viktor's models
flagged for July 2026**. If news does not reach the firms a relationship manager is about
to ring, that is the finding, and it is a stronger one because the sample is the deliverable
rather than a construction.

**Method is deliberately unchanged.** Same Guardian endpoint, same query construction, same
verification rules as `10_news_sentiment_model.ipynb`. Only the sample changes. If we
altered both at once, neither result would mean anything.

**Cost.** One API call per company, 398 in total, cached to disk. A free Guardian developer
key allows 500 calls a day and 12 a second, so this fits inside one day with room to retry.

## 1. Setup and the rate limit probe

In [ ]:
from pathlib import Path
from getpass import getpass
from html import unescape
import json, re, sys, time

import pandas as pd
import requests

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = Path("/content/drive/MyDrive/Lloyds")
else:
    WORK_DIR = Path("..").resolve() / "data" / "processed"

OUT_DIR = WORK_DIR / "dashboard_pack"
CACHE_DIR = WORK_DIR / "guardian_cache"
for d in (OUT_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

SHORTLIST = WORK_DIR / "shortlist_398_2026-07.csv"

SNAPSHOT_DATE = "2026-07-01"
FROM_DATE, TO_DATE = "2025-08-01", "2026-07-31"     # the 12 months before the base month
SECTIONS = "business|money|technology|uk-news"
PAGE_SIZE = 50
SHOW_FIELDS = "headline,standfirst,trailText,body,byline"
SHOW_TAGS = "keyword"
RATE_LIMIT_SECONDS = 0.5
GUARDIAN_URL = "https://content.guardianapis.com/search"

print("work dir :", WORK_DIR)
print("cache    :", CACHE_DIR)
print("window   :", FROM_DATE, "to", TO_DATE)

In [ ]:
GUARDIAN_API_KEY = getpass("Guardian API key: ").strip()

# The probe. One cheap call, before we spend any of the daily quota, so we find out what
# the allowance actually is rather than assuming it.
r = requests.get(GUARDIAN_URL, params={"q": "tesco", "page-size": 1,
                                       "api-key": GUARDIAN_API_KEY}, timeout=30)
print("status:", r.status_code)
interesting = [k for k in r.headers
               if any(w in k.lower() for w in ("rate", "limit", "quota", "remaining", "retry"))]
if interesting:
    for k in interesting:
        print(f"  {k}: {r.headers[k]}")
else:
    print("  the Guardian does not return rate limit headers on this tier.")
    print("  Published free developer allowance: 500 calls a day, 12 a second.")

if r.status_code == 200:
    print("\ncontrol query 'tesco' total hits:", r.json()["response"]["total"])
    print("the key works and the plumbing is live")
elif r.status_code == 429:
    print("\nAlready rate limited. Wait and re-run before going further.")
else:
    print("\nKey rejected:", r.text[:200])

## 2. The sample

`shortlist_398_2026-07.csv` is Viktor's four top-100 lists, deduplicated to 398 unique
companies, joined to Vishal's universe for the name, town and sector each search needs.

**27 of the 398 are not in the universe file.** Viktor scored the July snapshot and Vishal's
store is built from the June one, so those 27 are companies that entered the universe in
July. They are searched anyway using the name on the shortlist, and they are counted
separately, because a company the dashboard cannot display is worth knowing about.

In [ ]:
sample = pd.read_csv(SHORTLIST, dtype={"CompanyNumber": str})
print(f"companies: {len(sample):,}")
print(f"  in the universe file    : {sample['CompanyName'].notna().sum():,}")
print(f"  not in the universe file: {sample['CompanyName'].isna().sum():,}")
print()
print(sample.groupby("best_target").size().to_string())

# Clean names the same way the earlier notebook did, so query construction is identical.
_SUFFIX_RE = re.compile(r"\b(LIMITED|LTD|PLC|LLP|CIC|CIO|COMPANY|CO|HOLDINGS|GROUP)\b")

def clean_name(name):
    if name is None or (isinstance(name, float) and pd.isna(name)):
        return None
    s = re.sub(r"[^A-Za-z0-9 &]", " ", str(name).upper())
    s = _SUFFIX_RE.sub(" ", s)
    return re.sub(r"\s+", " ", s).strip() or None

sample["clean_name"] = sample["CompanyName"].map(clean_name)
print(f"\nsearchable names: {sample['clean_name'].notna().sum():,}")
sample[["CompanyNumber", "CompanyName", "clean_name", "post_town", "sector",
        "best_target"]].head(8)

## 3. Search, cached

In [ ]:
EMPTY = {"response": {"total": 0, "results": []}}

def build_query(clean):
    """Exact phrase. A bare multi-word name would match any article containing all the
    words anywhere, which is where most false positives came from last time."""
    if not clean or len(clean) < 3:
        return None
    return f'"{clean}"'


def guardian_search(company_number, clean, api_key, use_cache=True):
    """Returns (response, source). Never raises, so one odd name cannot halt the run."""
    cache_file = CACHE_DIR / f"{company_number}_guardian.json"
    if use_cache and cache_file.exists():
        return json.loads(cache_file.read_text(encoding="utf-8")), "cache"
    q = build_query(clean)
    if not q:
        return EMPTY, "skip"
    params = {"q": q, "query-fields": "body", "section": SECTIONS,
              "from-date": FROM_DATE, "to-date": TO_DATE, "lang": "en",
              "order-by": "relevance", "page-size": PAGE_SIZE,
              "show-fields": SHOW_FIELDS, "show-tags": SHOW_TAGS, "api-key": api_key}
    resp = requests.get(GUARDIAN_URL, params=params, timeout=30)
    if resp.status_code == 429:
        return EMPTY, "rate_limited"
    if resp.status_code != 200:
        return EMPTY, "error"
    data = resp.json()
    cache_file.write_text(json.dumps(data, ensure_ascii=False), encoding="utf-8")
    time.sleep(RATE_LIMIT_SECONDS)
    return data, "api"

In [ ]:
counts, sources = [], []
for i, row in enumerate(sample.itertuples(index=False), start=1):
    data, src = guardian_search(row.CompanyNumber, row.clean_name, GUARDIAN_API_KEY)
    counts.append(data["response"]["total"])
    sources.append(src)
    if src == "rate_limited":
        print(f"  rate limited at {i}/{len(sample)}. Re-run tomorrow, the cache resumes.")
        break
    if i % 50 == 0:
        print(f"  {i}/{len(sample)}  raw hits so far: {sum(counts):,}")

sample = sample.iloc[:len(counts)].copy()
sample["raw_hits"] = counts
sample["fetch"] = sources
print(f"\ndone: {len(sample):,} companies")
print(pd.Series(sources).value_counts().to_string())
print(f"\ncompanies with any raw hit: {(sample['raw_hits'] > 0).sum():,} of {len(sample):,}")
print(f"total raw articles        : {int(sample['raw_hits'].sum()):,}")

## 4. Verification

A raw hit means the Guardian's search engine found the phrase somewhere in an article body.
That is not the same as an article being **about** the company. The rules below are Vishal's,
carried over unchanged:

1. the name must appear as a standalone phrase, not inside another word or hyphenated
2. it must appear in the **summary** fields, not only buried in the body, because an
   article genuinely about a company names it up front
3. and it must be corroborated by the town, the sector, or business context words

A single common word as a company name cannot pass rule 1 on its own, which is why
"Baroness" returned Michelle Mone and "Coaster" returned roller coasters last time.

In [ ]:
BUSINESS_CONTEXT = ["company", "firm", "business", "ltd", "limited", "plc", "group",
                    "holdings", "ceo", "chief executive", "founder", "director",
                    "turnover", "revenue", "profit", "acquisition", "merger", "contract",
                    "customers", "employees", "staff", "headquarters"]


def _strip(blob):
    return unescape(re.sub(r"<[^>]+>", " ", blob)).lower()


def _summary_text(a):
    f = a.get("fields", {}) or {}
    return _strip(" ".join([a.get("webTitle", ""), f.get("headline", "") or "",
                            f.get("standfirst", "") or "", f.get("trailText", "") or ""]))


def _article_text(a):
    f = a.get("fields", {}) or {}
    return _summary_text(a) + " " + _strip(f.get("body", "") or "")


def _mentions(text, phrase):
    return re.search(r"(?<![\w-])" + re.escape(phrase) + r"(?![\w-])", text) is not None


def verify_article(a, clean, town, sector):
    name = (clean or "").lower().strip()
    summary, full = _summary_text(a), _article_text(a)
    phrase_in_summary = bool(name) and _mentions(summary, name)
    town_ok = bool(town) and isinstance(town, str) and _mentions(full, town.lower().strip())
    sector_ok = bool(sector) and isinstance(sector, str) and any(
        _mentions(full, w) for w in re.split(r"[^a-z]+", sector.lower()) if len(w) > 4)
    context_ok = any(_mentions(full, w) for w in BUSINESS_CONTEXT)
    single_word = len(name.split()) == 1
    corroborated = town_ok or sector_ok or (context_ok and not single_word)
    return {"verified": bool(phrase_in_summary and corroborated),
            "phrase": phrase_in_summary, "town": town_ok, "sector": sector_ok,
            "context": context_ok,
            "date": (a.get("webPublicationDate") or "")[:10],
            "title": a.get("webTitle", ""), "url": a.get("webUrl", "")}

In [ ]:
records = {}
for row in sample[sample["raw_hits"] > 0].itertuples(index=False):
    data, _ = guardian_search(row.CompanyNumber, row.clean_name, GUARDIAN_API_KEY)
    recs = [verify_article(a, row.clean_name, row.post_town, row.sector)
            for a in data["response"]["results"]]
    records[row.CompanyNumber] = recs
    n_keep = sum(r["verified"] for r in recs)
    if n_keep or len(recs) > 4:
        print(f"{row.clean_name[:38]:<40} {len(recs):>3} articles -> {n_keep} verified")

sample["n_verified"] = sample["CompanyNumber"].map(
    lambda cn: sum(r["verified"] for r in records.get(cn, [])))

print("\n" + "=" * 62)
print(f"companies searched            : {len(sample):,}")
print(f"companies with any raw hit    : {(sample['raw_hits'] > 0).sum():,}")
print(f"raw articles                  : {int(sample['raw_hits'].sum()):,}")
print(f"articles inspected            : {sum(len(v) for v in records.values()):,}")
print(f"articles passing verification : {int(sample['n_verified'].sum()):,}")
print(f"companies with verified news  : {(sample['n_verified'] > 0).sum():,}")

## 5. The control

A result of zero is only meaningful next to a positive control. If Tesco also returned
nothing, the conclusion would be that our code is broken, not that news does not reach SMEs.

In [ ]:
ctrl = requests.get(GUARDIAN_URL, params={
    "q": '"Tesco"', "query-fields": "body", "section": SECTIONS,
    "from-date": FROM_DATE, "to-date": TO_DATE, "page-size": 50,
    "show-fields": SHOW_FIELDS, "api-key": GUARDIAN_API_KEY}, timeout=30).json()

total = ctrl["response"]["total"]
recs = [verify_article(a, "TESCO", "WELWYN GARDEN CITY", "Retail")
        for a in ctrl["response"]["results"]]
print(f"Tesco, same window, same rules: {total:,} raw hits, "
      f"{sum(r['verified'] for r in recs)} of {len(recs)} inspected verified")
print("\nThe pipeline finds and verifies news when news exists.")

## 6. Write the pack

In [ ]:
sys.path.insert(0, str(WORK_DIR))
from dashboard_export import write_source_pack

rows = []
for cn, recs in records.items():
    for r in recs:
        if not r["verified"]:
            continue
        rows.append({"CompanyNumber": cn, "event_date": r["date"],
                     "event_type": "news_article", "detail": r["title"][:180],
                     "value": 1.0, "url": r["url"], "confidence": 0.7,
                     "match_method": "name_verified"})

events = pd.DataFrame(rows, columns=["CompanyNumber", "event_date", "event_type", "detail",
                                     "value", "url", "confidence", "match_method"])
print(f"verified news events: {len(events):,}")

if len(events):
    write_source_pack(events, prefix="news", source="guardian_news",
                      out_dir=OUT_DIR, snapshot_date=SNAPSHOT_DATE)
else:
    print("\nNo verified news events, so there is no events file to write.")
    print("That is the result, not a failure. Section 7 records it properly.")

## 7. The coverage table for the report

Whatever the numbers are, this table is the deliverable. A measured zero on 398 companies
with a working control is a stronger statement than a measured zero on 96.

In [ ]:
searched = len(sample)
summary = pd.DataFrame([
    {"stage": "companies searched", "n": searched, "pct": "100%"},
    {"stage": "any raw hit", "n": int((sample["raw_hits"] > 0).sum()),
     "pct": f"{(sample['raw_hits'] > 0).mean():.1%}"},
    {"stage": "any verified article", "n": int((sample["n_verified"] > 0).sum()),
     "pct": f"{(sample['n_verified'] > 0).mean():.1%}"},
])
print(summary.to_string(index=False))

by_target = sample.groupby("best_target").agg(
    companies=("CompanyNumber", "size"),
    raw_hits=("raw_hits", "sum"),
    with_raw=("raw_hits", lambda s: int((s > 0).sum())),
    with_verified=("n_verified", lambda s: int((s > 0).sum()))).reset_index()
print("\nby model target:")
print(by_target.to_string(index=False))

out = OUT_DIR / "news_coverage_summary.csv"
sample[["CompanyNumber", "CompanyName", "best_target", "best_rank", "best_score",
        "sector", "post_town", "raw_hits", "n_verified", "fetch"]].to_csv(out, index=False)
print(f"\nsaved {out}")

## 8. What to write down

Fill these in once it has run:

- companies searched, raw hit rate, verified rate
- the Tesco control number, to show the plumbing works
- the collisions worth naming, since concrete examples land better than a percentage
- and the sentence the chapter turns on: the statutory record reaches small companies
  completely but only once something has gone wrong, while journalism does not reach them
  at all. Neither is a source of early warning, and that is the honest answer to the
  question the project asked.